In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import os



data_path = r"C:\workspace\NaverBoostCamp\Book Rating Prediction\data"
users = pd.read_csv(os.path.join(data_path, 'users.csv'))
users = pd.read_csv(os.path.join(data_path, 'users.csv'))
books = pd.read_csv(os.path.join(data_path, 'books.csv'))
interactions = pd.read_csv(os.path.join(data_path, 'train_ratings.csv'))

print(f"Users: {len(users)}")
print(f"Books: {len(books)}")
print(f"Interactions: {len(interactions)}")

In [ ]:
users.head(20)

In [ ]:
interactions.head(20)

In [ ]:
users['location_1'] = users['location'].str.split(',').str[0].str.strip()
users['location_1'] = users['location_1'].replace('', 'Unknown').fillna('Unknown')

In [ ]:
users['age_group'] = users['age'].apply(lambda x:
    'Unknown' if pd.isna(x) else
    '10s' if x < 20 else
    '20s' if x < 30 else
    '30s' if x < 40 else
    '40s' if x < 50 else '50+')

books['year_group'] = books['year_of_publication'].apply(lambda x:
    'Unknown' if pd.isna(x) else
    '~1960' if x < 1960 else
    '1960s' if x < 1970 else
    '1970s' if x < 1980 else
    '1980s' if x < 1990 else
    '1990s' if x < 2000 else
    '2000s' if x < 2010 else
    '2010s' if x < 2020 else
    '2020+')

In [ ]:

# Books: 카테고리형 변수 top 15만
for col in ['book_author', 'category', 'publisher']:
    top_15 = books[col].value_counts().nlargest(15).index
    books[f'{col}_top15'] = books[col].apply(lambda x: x if x in top_15 else 'Other')

# location_1도 top 15만!
top_15_locations = users['location_1'].value_counts().nlargest(15).index
users['location_1_top15'] = users['location_1'].apply(lambda x: x if x in top_15_locations else 'Other')

# === 2. 조인 ===
df = interactions.merge(users[['user_id', 'location_1_top15', 'age', 'age_group']], on='user_id', how='left')
df = df.merge(books[['isbn', 'book_author_top15', 'category_top15', 'publisher_top15', 'year_of_publication', 'year_group']],
              on='isbn', how='left')


print(f"\n조인 후 데이터: {len(df)}")
print("\n=== 컬럼 정보 ===")
print(df.columns.tolist())

In [ ]:
import matplotlib.patches as mpatches
import numpy as np
import sys
from io import StringIO
import os

def analyze_interaction(df, feature1, feature2, is_f1_categorical=True, is_f2_categorical=True,
                       n_bins=10, min_samples=5, global_vmax=1.0, exclude_other=True,
                       significance_threshold=0.25, save_dir=None):
    """
    Parameters:
    -----------
    exclude_other : bool
        'Other' 카테고리 제외 여부 (기본 True)
    significance_threshold : float
        유의미한 상호작용으로 간주할 절댓값 임계값 (기본 0.25)
    save_dir : str, optional
        결과 저장 디렉토리 (예: './results')
    """

    # 초기화

    text_capture = None
    old_stdout = None
    img_path = None
    txt_path = None

    # 저장 설정
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        safe_name = f"{feature1}_vs_{feature2}".replace('/', '_')
        img_path = os.path.join(save_dir, f"{safe_name}.png")
        txt_path = os.path.join(save_dir, f"{safe_name}.txt")
        old_stdout = sys.stdout
        text_capture = StringIO()
        sys.stdout = text_capture

    df_clean = df[[feature1, feature2, 'rating']].dropna().copy()
    before_len = len(df_clean)
    after_len = len(df_clean)
    # Other 제외
    if exclude_other:
        df_clean = df_clean[
            (df_clean[feature1] != 'Other') &
            (df_clean[feature2] != 'Other')
        ]
        after_len = len(df_clean)

    f1_name = feature1
    f2_name = feature2

    if not is_f1_categorical:
        df_clean[f'{feature1}_bin'] = pd.qcut(df_clean[feature1], q=n_bins, duplicates='drop')
        f1_name = f'{feature1}_bin'
    if not is_f2_categorical:
        df_clean[f'{feature2}_bin'] = pd.qcut(df_clean[feature2], q=n_bins, duplicates='drop')
        f2_name = f'{feature2}_bin'

    overall_mean = df_clean['rating'].mean()

    f1_mean = df_clean.groupby(f1_name, observed=True)['rating'].mean()
    f2_mean = df_clean.groupby(f2_name, observed=True)['rating'].mean()

    actual = df_clean.pivot_table(values='rating', index=f2_name, columns=f1_name, aggfunc='mean', observed=True)
    counts = df_clean.pivot_table(values='rating', index=f2_name, columns=f1_name, aggfunc='count', observed=True)

    expected = pd.DataFrame(index=actual.index, columns=actual.columns, dtype=float)
    for f1 in actual.columns:
        for f2 in actual.index:
            expected.loc[f2, f1] = overall_mean + (f1_mean[f1] - overall_mean) + (f2_mean[f2] - overall_mean)

    interaction = actual - expected

    # ========== 데이터 적은 셀은 NaN 처리 ==========
    mask_low = counts < min_samples

    actual_filtered = actual.copy()
    actual_filtered[mask_low] = np.nan

    expected_filtered = expected.copy()
    expected_filtered[mask_low] = np.nan

    interaction_filtered = interaction.copy()
    interaction_filtered[mask_low] = np.nan

    # 상호작용 크기 (절댓값)
    interaction_magnitude = abs(interaction_filtered)

    counts_display = counts.copy()
    counts_display[mask_low] = np.nan

    # 시각화
    fig, axes = plt.subplots(2, 2, figsize=(18, 14))

    # 1. Actual
    sns.heatmap(actual_filtered, annot=False, fmt='.2f', cmap='RdYlGn', ax=axes[0, 0],
                vmin=df_clean['rating'].min(), vmax=df_clean['rating'].max(),
                cbar_kws={'label': 'Rating'})
    axes[0, 0].set_title(f'Actual Mean Rating (n>={min_samples})', fontsize=12, fontweight='bold')

    # 2. Expected
    sns.heatmap(expected_filtered, annot=False, fmt='.2f', cmap='RdYlGn', ax=axes[0, 1],
                vmin=df_clean['rating'].min(), vmax=df_clean['rating'].max(),
                cbar_kws={'label': 'Rating'})
    axes[0, 1].set_title(f'Expected (No Interaction, n>={min_samples})', fontsize=12, fontweight='bold')

    # 3. Interaction Magnitude (절댓값) - 고정 스케일
    sns.heatmap(interaction_magnitude, annot=False, fmt='.2f', cmap='Reds', ax=axes[1, 0],
                vmin=0, vmax=global_vmax,
                cbar_kws={'label': 'Interaction Magnitude'})
    axes[1, 0].set_title(f'Interaction Magnitude (scale: 0-{global_vmax}, n>={min_samples})',
                        fontsize=12, fontweight='bold')

    # 4. Counts
    from matplotlib.colors import LogNorm
    sns.heatmap(counts_display, annot=False, fmt='d', cmap='Blues', ax=axes[1, 1],
                norm=LogNorm(vmin=1, vmax=counts.max().max()),
                cbar_kws={'label': 'Count (log scale)'})
    axes[1, 1].set_title(f'Sample Counts (showing n>={min_samples} only)', fontsize=12, fontweight='bold')

    plt.tight_layout()

    # 이미지 저장
    if save_dir:
        plt.savefig(img_path, dpi=150, bbox_inches='tight')

    plt.show()

    # ========== 통계 ==========
    interaction_flat = interaction_filtered.unstack().dropna()

    total_cells = len(actual.unstack())
    valid_cells = len(interaction_flat)
    filtered_cells = total_cells - valid_cells
    print(f"\n{'='*80}")
    print(f"분석: {feature1} vs {feature2}")
    print(f"{'='*80}")

    if exclude_other:
        print(f"'Other' 제외: {before_len} → {after_len}개 ({before_len - after_len}개 제거)")

    print(f"유효 데이터: {len(df_clean)}개")
    print(f"전체 평균 평점: {overall_mean:.3f}")

    print(f"\n전체 셀: {total_cells}개 | 유효 셀: {valid_cells}개 | 필터링됨: {filtered_cells}개")

    if len(interaction_flat) > 0:
        # ========== 피쳐 간 전체 상호작용 임팩트 ==========
        print(f"\n{'='*80}")
        print(f"피쳐 간 상호작용 임팩트 (Feature Interaction Impact)")
        print(f"{'='*80}")

        # Main Effects 편차
        f1_main_effect = f1_mean - overall_mean
        f2_main_effect = f2_mean - overall_mean

        # 각 카테고리별 샘플 수
        f1_counts = df_clean.groupby(f1_name, observed=True).size()
        f2_counts = df_clean.groupby(f2_name, observed=True).size()

        # 1. Main Effects 크기 (가중 RMS)
        f1_effect = np.sqrt(((f1_main_effect**2) * f1_counts).sum() / f1_counts.sum())
        f2_effect = np.sqrt(((f2_main_effect**2) * f2_counts).sum() / f2_counts.sum())

        # 2. Interaction Effect 크기 (가중 RMS)
        # 유효한 셀만 사용 (min_samples 이상)
        valid_mask = ~mask_low
        interaction_valid = interaction[valid_mask]
        counts_valid = counts[valid_mask]

        # 가중 RMS
        weighted_sq_sum = ((interaction_valid**2) * counts_valid).sum().sum()
        total_count = counts_valid.sum().sum()
        interaction_strength = np.sqrt(weighted_sq_sum / total_count)

        # 3. 설명력 (제곱합 기준) - 이미 가중치 적용됨
        total_variance = ((actual - overall_mean) ** 2 * counts).sum().sum()
        interaction_variance = ((interaction ** 2)[valid_mask] * counts[valid_mask]).sum().sum()
        explained_ratio = interaction_variance / total_variance if total_variance > 0 else 0

        print(f"\n1. Main Effects (전체 평균치에서)(가중 RMS):")
        print(f"   {feature1}: {f1_effect:.3f}")
        print(f"   {feature2}: {f2_effect:.3f}")
        print(f"   합계: {f1_effect + f2_effect:.3f}")

        print(f"\n2. Interaction Effect (기대치에서) (가중 RMS):")
        print(f"   RMS: {interaction_strength:.3f}")
        print(f"   최대: {abs(interaction_flat).max():.3f}")
        print(f"   가중 평균: {(abs(interaction_valid) * counts_valid).sum().sum() / total_count:.3f}")

        print(f"\n3. 설명력 (분산 기준):")
        print(f"   상호작용이 전체 분산의 {explained_ratio*100:.1f}% 설명")

        # ========== 4. 집중도 분석 ==========
        print(f"\n4. 집중도 분석:")

        # 유의미한 상호작용만 추출
        significant_mask = abs(interaction_filtered) > significance_threshold
        significant_interactions = interaction[significant_mask]
        significant_counts = counts[significant_mask]

        n_significant = significant_mask.sum().sum()
        coverage = n_significant / valid_cells if valid_cells > 0 else 0

        print(f"   유의미한 셀 (|interaction| > {significance_threshold}): {n_significant}개 ({coverage*100:.1f}%)")

        if n_significant > 1:
            # 가중 평균과 표준편차
            sig_total_count = significant_counts.sum().sum()
            weighted_mean = (significant_interactions.abs() * significant_counts).sum().sum() / sig_total_count

            # 가중 분산 및 표준편차
            weighted_variance = (((significant_interactions.abs() - weighted_mean)**2 * significant_counts).sum().sum()
                                / sig_total_count)
            weighted_std = np.sqrt(weighted_variance)

            print(f"   가중 표준편차: {weighted_std:.3f}")
            print(f"   가중 분산: {weighted_variance:.3f}")

            if coverage < 0.2:
                print(f"   ⚠️  Coverage 낮음 ({coverage*100:.1f}%) - 대부분 셀은 상호작용 약함")


    if save_dir and text_capture is not None:
        sys.stdout = old_stdout
        text_result = text_capture.getvalue()
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(text_result)
        print(text_result)

In [ ]:

# 1. 장르 - 지역
analyze_interaction(df, 'category_top15', 'location_1_top15',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

# 2. 장르 - 나이
analyze_interaction(df, 'category_top15', 'age_group',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

# 3. 장르 - 출판 연도
analyze_interaction(df, 'category_top15', 'year_group',
                       is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

# 4. 장르 - 저자
analyze_interaction(df, 'category_top15', 'book_author_top15',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

In [ ]:
# 5. 지역 vs 나이
analyze_interaction(df, 'location_1_top15', 'age_group',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

# 6. 지역 vs 출판 연도
analyze_interaction(df, 'location_1_top15', 'year_group',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

# 7. 지역 vs 저자
analyze_interaction(df, 'location_1_top15', 'book_author_top15',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

In [ ]:
# 8. 나이 vs 출판 연도
analyze_interaction(df, 'age_group', 'year_group',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

# 9. 나이 vs 저자
analyze_interaction(df, 'age_group', 'book_author_top15',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')

In [ ]:

# 10. 출판 연도 vs 저자
analyze_interaction(df, 'year_group', 'book_author_top15',
                   is_f1_categorical=True, is_f2_categorical=True,
                   global_vmax=1.0, exclude_other=True, significance_threshold=0.15, save_dir='./results')